In [ ]:
import polars as pl

csv_path = "output/all_trials_timeseries.csv"
df = pl.read_csv(csv_path, infer_schema_length=10000)

trial_df = (
    df.select(["subject", "velocity", "trial", "step_TF"])
    .drop_nulls(["subject", "velocity", "trial", "step_TF"])
    .unique(subset=["subject", "velocity", "trial"], keep="first")
    .filter(pl.col("step_TF").is_in(["step", "nonstep"]))
)

summary_step_ratio = (
    trial_df.group_by("step_TF")
    .len()
    .rename({"len": "trial_count"})
    .with_columns(
        (pl.col("trial_count") / pl.sum("trial_count") * 100).round(2).alias("ratio_percent")
    )
    .sort("step_TF")
)

subject_step_counts = (
    trial_df.group_by(["subject", "step_TF"])
    .len()
    .rename({"len": "trial_count"})
    .sort(["subject", "step_TF"])
)

summary_subject_mean_trials = (
    subject_step_counts.group_by("step_TF")
    .agg(pl.col("trial_count").mean().round(2).alias("mean_trial_count_per_subject"))
    .sort("step_TF")
)

print("[1] step/nonstep trial count & ratio")
print(summary_step_ratio)

print("\n[2] mean trial count per subject (by step/nonstep)")
print(summary_subject_mean_trials)

print("\n[debug] subject x step trial counts")
print(subject_step_counts)
